In [1]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%%capture
!pip install datasets transformers evaluate rouge_score accelerate, relplot
!pip install git+https://github.com/google-research/bleurt.git
!pip install --upgrade bitsandbytes # numpy pandas
#!pip install unbabel-comet


In [3]:
!pip -q install relplot evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00


In [4]:
!rm -rf colab_llm_utils

In [5]:
!git clone -b multiaxial https://github.com/ravy101/colab_llm_utils.git

Cloning into 'colab_llm_utils'...
remote: Enumerating objects: 1192, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 1192 (delta 61), reused 68 (delta 32), pack-reused 1092 (from 1)
Receiving objects: 100% (1192/1192), 171.50 KiB | 2.38 MiB/s, done.
Resolving deltas: 100% (750/750), done.


In [6]:
import colab_llm_utils
from colab_llm_utils import configs

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, AutoConfig, BitsAndBytesConfig, GenerationConfig
from accelerate import infer_auto_device_map, init_empty_weights

In [8]:

import time
import re
import sys
import os
import string
import math

In [9]:
import types
import numpy.core as np_core
import numpy as np

np._core = types.ModuleType("_core")
np._core.__dict__.update(np_core.__dict__)

In [10]:
import torch
from torch.nn import functional as F
import evaluate

In [11]:
dataset = colab_llm_utils.configs.datasets.mmlu

model_config = colab_llm_utils.configs.models.qwen3_8b
large_model_config = colab_llm_utils.configs.models.qwen3_8b

#embedding_model_config = colab_llm_utils.configs.models.t5_base
short_name = model_config['model_name'].split('/')[-1]


special_tag = ''
special_tag_large = 'rag'



FOLLOW_UP = True
PTRUE = False
SEMDEC = False

In [12]:
DICT_ANS = dataset['dict_ans']
SELF_CONF = False
SKIP_PROCESSING_SMALL = True
SKIP_PROCESSING_LARGE = False
INDEX_JOIN = True
SKIP_MERGING = True
drive_path = f"/content/drive/MyDrive/phase3/Llama/{dataset['clean_name']}/"

In [13]:
LARGE_THINKING = True
if LARGE_THINKING:
  special_tag_large = "thinking"

In [14]:

if PTRUE:
  short_name = short_name + "ptrue"
  special_tag = "p_true"


if SELF_CONF:
  short_name = short_name + "confconf"
  special_tag = "self_conf"


if SEMDEC:
  short_name = short_name + "_semlexdec"

In [16]:
metric_dict = {}
if dataset['task_type'] == 'translation':
  metric_dict['meteor'] = colab_llm_utils.scorers.get_meteor()
  metric_dict['bleurt'] = colab_llm_utils.scorers.get_bleurt()
  metric_dict['rouge'] = colab_llm_utils.scorers.get_rouge()
  #comet = colab_llm_utils.scorers.get_comet()
elif dataset['task_type'] == 'summarization':
  metric_dict['rouge'] = colab_llm_utils.scorers.get_rouge()

# Load Intermediate Data Files

In [17]:
files = os.listdir(drive_path)
files.sort(key=lambda x: x) #name sort
files.reverse()

In [18]:
files.reverse()

In [19]:

small_results_7 = []
results_13 = []
results_70 = []


In [20]:
short_name

'Qwen3-8B'

In [21]:
files

['allQwen3-8B.pickle',
 'allQwen3-8Brag.pickle',
 'allQwen3-8Bthinking.pickle',
 'all_auxiliary_train_Qwen3-8B_0010.pickle',
 'all_auxiliary_train_Qwen3-8B_0011.pickle',
 'all_auxiliary_train_Qwen3-8B_0012.pickle',
 'all_auxiliary_train_Qwen3-8B_0013.pickle',
 'all_auxiliary_train_Qwen3-8B_0014.pickle',
 'all_auxiliary_train_Qwen3-8B_0015.pickle',
 'all_auxiliary_train_Qwen3-8B_0016.pickle',
 'all_auxiliary_train_Qwen3-8B_0017.pickle',
 'all_auxiliary_train_Qwen3-8B_0018.pickle',
 'all_auxiliary_train_Qwen3-8B_0019.pickle',
 'all_auxiliary_train_Qwen3-8B_0020.pickle',
 'all_auxiliary_train_Qwen3-8B_0021.pickle',
 'all_auxiliary_train_Qwen3-8B_0022.pickle',
 'all_auxiliary_train_Qwen3-8B_0023.pickle',
 'all_auxiliary_train_Qwen3-8B_0024.pickle',
 'all_auxiliary_train_Qwen3-8B_0025.pickle',
 'all_auxiliary_train_Qwen3-8B_0026.pickle',
 'all_auxiliary_train_Qwen3-8B_0027.pickle',
 'all_auxiliary_train_Qwen3-8B_0028.pickle',
 'all_auxiliary_train_Qwen3-8B_0029.pickle',
 'all_auxiliary_trai

In [22]:
f"{short_name}{special_tag}" + "_"

'Qwen3-8B_'

In [23]:
str(dataset["dataset_name"])

'all'

In [24]:
file_limit = 100
if not SKIP_PROCESSING_SMALL:
  results_7 = []
  for f in files:
    if "small" in f:
      continue
    if str(dataset["dataset_name"]) + "_" not in f:
      continue
    #print(f)
    if len(results_7) >= file_limit:
      break
    if f.endswith(".pickle") and dataset["subset"] in f and f"{short_name}{special_tag}" + "_"  in f:
      print(f"**********************************reading {f}")
      start_time = time.perf_counter()
      df = pd.read_pickle(os.path.join(drive_path, f))
      df = colab_llm_utils.data_processing.process_dataframe(df, dataset, metric_dict, self_conf = SELF_CONF, p_true = PTRUE, thinking = False)
      results_7.append(df)
      end_time = time.perf_counter()

      elapsed_time = end_time - start_time
      print(f"Execution time: {elapsed_time:.4f} seconds")

  res_7 = colab_llm_utils.data_processing.combine_dataframe(results_7)
  if FOLLOW_UP:
    colab_llm_utils.data_processing.columnize_meta_field(res_7, 'follow_up')
    res_7['def_axis'] = [colab_llm_utils.data_processing.coerce_to_bounded_int(out) for out in res_7["follow_up-text"]]
  res_7.to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}.pickle"))
else:
  res_7 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}.pickle"))

In [25]:
large_tag = large_model_config['model_name'].split('/')[-1] + special_tag_large
if not SKIP_PROCESSING_LARGE:
  results_13 = []
  for f in files:
    if "small" in f:
      continue
    if str(dataset["dataset_name"]) + "_" not in f:
      continue
    if len(results_13) >= file_limit:
      break
    if f.endswith(".pickle") and dataset["subset"] in f and large_tag  + "_" in f:
      print(f"********************************************reading {f}")
      start_time = time.perf_counter()
      df = pd.read_pickle(os.path.join(drive_path, f))
      df = colab_llm_utils.data_processing.process_dataframe(df, dataset, metric_dict, self_conf=False, p_true = False, thinking=LARGE_THINKING)
      results_13.append(df)
      end_time = time.perf_counter()

      elapsed_time = end_time - start_time
      print(f"Execution time: {elapsed_time:.4f} seconds")
  res_13 = colab_llm_utils.data_processing.combine_dataframe(results_13)
  res_13.to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{large_tag}.pickle"))
else:
  try:
    res_13 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{large_tag}.pickle"))
  except:
    print("adding empty large.")
    res_13 = res_7[res_7['all_probas'] == False]

********************************************reading all_test_Qwen3-8Bthinking_0000.pickle
missing index 2790
in candidates [4802, 19420, 51350, 14806, 25691, 12060, 10578, 9142, 40253, 45306]
Loading ROUGE metric...


Execution time: 7.2867 seconds
********************************************reading all_test_Qwen3-8Bthinking_0001.pickle
Execution time: 10.2114 seconds
********************************************reading all_test_Qwen3-8Bthinking_0002.pickle
missing index 15888
in candidates [27262, 19805, 8045, 11677, 264, 4541, 74616, 56283, 23009, 2041]
missing index 6959
in candidates [330, 3259, 421, 2272, 72867, 2999, 20102, 279, 8365, 566]
Execution time: 4.1561 seconds
********************************************reading all_test_Qwen3-8Bthinking_0003.pickle
missing index 3196
in candidates [279, 66538, 330, 3619, 20836, 51136, 89587, 20601, 1179, 5506]
missing index 3491
in candidates [2606, 2856, 1319, 1887, 60227, 5114, 16523, 3405, 1465, 40202]
missing index 10922
in candidates [3491, 2309, 2456, 5114, 3405, 9023, 943, 4647, 6286, 330]
missing index 4115
in candidates [2003, 2494, 6460, 46629, 882, 4428, 2790, 6625, 4168, 279]
missing index 894
in candidates [264, 49098, 7945, 9481, 6825, 1

In [26]:
if not SKIP_MERGING:
  if INDEX_JOIN:
    try:
      res_13 = res_13.drop(["ans"], axis=1)
    except:
      pass
    full_res = pd.merge(res_7, res_13, how='left', left_index=True, right_index=True, suffixes=(None,'_large'))
  else:
    full_res = pd.merge(res_7, res_13.drop(["ans"], axis=1), how='left', left_on='prompts', right_on='prompts', suffixes=(None,'_large'))
  #full_res['bleurt_13b'] = [b[0] for b in full_res['bleurt_13b']]
  if SELF_CONF:
    full_res['self_conf'] = [int(c.split("Confidence: ")[-1].strip()[:-1])/100 for c in full_res['self_conf']]
  full_res.to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}_full.pickle"))
#else:
#  full_res = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}_full.pickle"))

In [27]:
from datetime import datetime
print(f"all done {datetime.now().strftime("%H:%M:%S")}")

all done 00:37:35


In [28]:
res_7['f1'].mean()

np.float64(0.439)

In [29]:
res_7['follow_up-text']

,follow_up-text
0,1\n\n
1,2\n\n
2,1\n\n
3,1\n\n
4,1\n\n
...,...
1995,1\n\n
1996,1\n\n
1997,1\n\n
1998,1\n\n
